# Phase 0: 키워드 발견 및 확장 (Keyword Discovery)

이 노트북은 `sample_for_claude.py` 를 활용하여 **포화 기반 키워드 발견 워크플로우**를 실행합니다.

## 워크플로우 요약

```
01_cleaned_merged.csv (정제된 원본 데이터)
  → 씨앗 키워드(seed_keywords.yaml)로 후보군 필터링
    → 구별 층화 랜덤 샘플링 (1,000건/라운드)
      → Claude Code에 프롬프트 전달 → 정밀 키워드 발견
        → 포화(신규 키워드 < 5%)까지 반복
          → keywords.yaml 저장
```

## 1단계: 랜덤 샘플 추출

아래 셀을 실행하면:
1. `data/processed/01_cleaned_merged.csv`에서 씨앗 키워드 포함 후보군 필터링
2. 25개 구에서 균등하게 층화 랜덤 샘플링
3. `data/processed/sample_round{N}.csv` 와 `data/processed/prompt_round{N}.md` 저장

> ⚠️ `--round` 번호를 라운드마다 올려주세요.

In [ ]:
# 라운드 번호 설정 (매 라운드마다 1씩 증가)
ROUND = 1
N_PER_ROUND = 1000  # 라운드당 샘플 수

In [ ]:
!cd .. && uv run python -m src.sample_for_claude sample --round {ROUND} --n-per-round {N_PER_ROUND}

## 2단계: Claude Code에 프롬프트 전달

위에서 생성된 `data/processed/prompt_round{N}.md` 파일을 열어 내용을 **Claude Code**에 전달합니다.

### 전달 방법

1. `prompt_round{N}.md` 파일을 열어 내용을 복사
2. Claude Code 세션에 붙여넣기
3. Claude가 JSON 형식으로 키워드를 응답
4. 응답 JSON을 `data/processed/claude_response_round{N}.json`으로 저장

> 💡 프롬프트에는 최대 50건의 게시글이 포함됩니다. 전체 샘플이 필요한 경우 `sample_round{N}.csv`를 직접 Claude에 전달하세요.

## 3단계: Claude 응답 병합 + 포화도 체크

Claude의 JSON 응답을 `keywords.yaml`에 병합하고, 포화 여부를 판단합니다.

In [ ]:
!cd .. && uv run python -m src.sample_for_claude merge --round {ROUND} --input data/processed/claude_response_round{ROUND}.json

## 상태 확인

현재까지의 키워드 발견 진행 상태를 확인합니다.

In [ ]:
!cd .. && uv run python -m src.sample_for_claude status

## 포화 판단 기준

| 라운드 | 신규 키워드 발견율 | 판단 |
|--------|-------------------|------|
| 1차 | - | 기준선 생성 |
| 2차 | ≥ 20% | 계속 |
| 3차 | 5~20% | 계속 |
| 4차~ | < 5% | **포화 → 종료** |

### 포화 후 다음 단계

포화가 확인되면 `config/keywords.yaml`에 정밀 키워드가 저장되어 있습니다.  
→ Phase 1 (키워드 필터링: `src/filter_by_keyword.py`)으로 진행합니다.